
# Tutorial 2 — `acquisition.py` (running a measurement)

**Prerequisite:** read `hardware_tutorial.ipynb` (especially TimeTagger install).

---

## What is `acquisition.py`?

If `hardware.py` is the **remote control** for each instrument, `acquisition.py` is the **experiment script** that:

1. Creates a folder for today's data
2. Connects & configures the TimeTagger
3. Optionally moves the rotation stage
4. Records for some duration
5. Repeats for many powers / angles
6. Cleans up

It does **not** know g² physics. It asks a pluggable **`Recorder`**: "what should I save for each run?"

```
Acquisition (conductor)
    ├── TimeTaggerDevice  (from hardware.py)
    ├── RotationStageController  (optional)
    └── Recorder  (you choose)
            ├── RawTimeTagRecorder  → saves .ttbin (raw tags)
            └── CorrelationRecorder → saves .pkl (see measurement tutorial)
```



# Bootstrap — run this cell first

This finds the project folder and lets Python import our code from `src/`.

import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src" / "hardware.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)


In [ ]:

# Bootstrap — run this cell first

This finds the project folder and lets Python import our code from `src/`.

import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src" / "hardware.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)


In [ ]:

## Lab vs laptop

| Flag | When to use |
|------|-------------|
| `LIVE_HARDWARE = False` | At home, on GitHub, learning the API (default) |
| `LIVE_HARDWARE = True` | On the **lab PC**, instruments plugged in |

Set it in the next cell before running hardware cells.

LIVE_HARDWARE = False   # <-- change to True on the lab PC

# --- edit these for YOUR bench ---
TT_SERIAL = ""              # TimeTagger serial, or "" for first device
PRM1_SERIAL = "27264707"    # Thorlabs K-Cube serial (string)
ELL14_ADDRESS = 2           # Elliptec address (integer 0-9)
CHANNELS = [1, 2, 3, 4, 5, 6]



---
# Part A — Configuration objects (the "forms" you fill in)

All settings are **dataclasses** — plain containers with typed fields and defaults.

## `LaserParams` — metadata about the laser (not controlled here)

| Field | Default | Unit | Meaning |
|-------|---------|------|---------|
| `rep_rate_hz` | 21e6 | Hz | Pulse repetition rate |
| `wavelength_nm` | 2100 | nm | Wavelength |

## `TimeTaggerParams` — everything about channels & histogram settings

| Field | Default | Meaning |
|-------|---------|---------|
| `tt_mode` | `"Standard"` | Resolution mode |
| `channels` | `[1..6]` | Which inputs to use |
| `mode_on_channel` | `H3T, H3R, ...` | Labels (for analysis) |
| `trigger_levels_v` | `0.5` | Threshold V (scalar or list) |
| `deadtime_ps` | 6× `[0]` | Dead time per channel |
| `delays_ps` | 6× `[0]` | Delay per channel |
| `binwidth_ps` | `100` | Histogram bin width (for correlation recorder) |
| `num_bins` | `5000` | Histogram length |
| `coincidence_window_ps` | `1000` | Coincidence window |
| `trigger`, `filter` | `[]` | Conditional filter channels |

`.to_schema()` converts this to the dict format used in saved files.

## `ChunkingParams` — "measure until good enough"

| Field | Default | Meaning |
|-------|---------|---------|
| `enabled` | `False` | Split run into chunks? |
| `chunk_minutes` | `2.0` | Length of each chunk |
| `max_chunks` | `100` | Safety cap |
| `check_every_n_chunks` | `1` | How often to check stop condition |
| `stop_when_reached` | `True` | Stop early if condition met |

## `AcquisitionConfig` — top-level settings

| Field | Meaning |
|-------|---------|
| `base_dir` | Where run folders are created |
| `material` | Sample name (e.g. `"CdTe110"`) |
| `experiment_type` | Tag in folder name (`"raw"`, `"g2_heralded_virtual"`, …) |
| `laser`, `timetagging`, `chunking` | Nested configs above |
| `repeats` | How many times to repeat full scan |
| `tt_serial` | Specific tagger serial (`""` = any) |

## `ScanPoint` — one row in a power/angle scan

| Field | Meaning |
|-------|---------|
| `power_mw` | Laser power (metadata, for filenames & analysis) |
| `angle_deg` | HWP angle to move to before acquiring |
| `stage_id` | Which stage to move (`"27264707"`) |
| `label` | Optional custom filename |


In [ ]:

from src.acquisition import (
    LaserParams, TimeTaggerParams, ChunkingParams,
    AcquisitionConfig, ScanPoint,
)

cfg = AcquisitionConfig(
    base_dir=ROOT / "data" / "tutorial_runs",
    material="CdTe110",
    experiment_type="raw",
    laser=LaserParams(rep_rate_hz=18.66e6, wavelength_nm=2100),
    timetagging=TimeTaggerParams(
        channels=[1, 2, 3, 4, 5, 6],
        trigger_levels_v=0.5,
        delays_ps=[0, -1000, 20000, 16500, 17800, 17300],
    ),
    chunking=ChunkingParams(enabled=False, chunk_minutes=1.0),
)
print("Config OK. experiment_type =", cfg.experiment_type)
print("timetagging schema keys:", list(cfg.timetagging.to_schema().keys()))



---
# Part B — The `Recorder` pattern (important idea)

`Acquisition` never decides *what* to save. It calls:

```python
recorder.record(device, duration_ps, savepath, params=..., logger=...)
```

| Recorder | Class | Output | When to use |
|----------|-------|--------|-------------|
| Raw tags | `RawTimeTagRecorder` | `.ttbin` + `.meta.pkl` | Maximum flexibility; analyse offline |
| Live g² | `CorrelationRecorder` | `.pkl` histograms | Smaller files; same format as old lab scripts |

Both implement the same interface → swap by passing `recorder=` to `Acquisition`.

### `RawTimeTagRecorder(max_file_size_mb=100, record_countrate=True)`

| Parameter | Meaning |
|-----------|---------|
| `max_file_size_mb` | Split `.ttbin` when file exceeds this size (0 = no split) |
| `record_countrate` | Also measure singles during raw recording |

### `Recorder.merge(accumulated, new)` 

Used internally when **chunking**: adds counts, sums histograms, averages rates. Override in `CorrelationRecorder` for rich data.


In [ ]:

from src.acquisition import RawTimeTagRecorder, Recorder
import inspect

print("RawTimeTagRecorder signature:", inspect.signature(RawTimeTagRecorder.__init__))
r = RawTimeTagRecorder(max_file_size_mb=100, record_countrate=True)
print("name:", r.name)



---
# Part C — `Acquisition` class (the conductor)

## Constructor

```python
Acquisition(
    config,                      # AcquisitionConfig (required)
    recorder=None,               # default: RawTimeTagRecorder()
    device=None,                   # default: built from config
    stages=None,                   # optional RotationStageController
    stop_condition=None,           # optional early-stop function
)
```

On creation it already:
- Makes a timestamped folder under `base_dir`
- Creates a log file inside that folder
- Builds `_general_params` (saved to `general_parameters.json` on setup)

## Methods you call

| Method | When | What happens |
|--------|------|--------------|
| `setup()` | Start | Connect tagger, apply triggers/delays/deadtimes |
| `characterize(label, duration_s, prompt=None)` | Optional | Measure count rates (cover/uncover beam) |
| `run_point(point, duration_s)` | Single run | One acquisition of fixed length |
| `run_chunked(point)` | Long run | Many chunks + merge + optional early stop |
| `run_scan(points)` | Power scan | Loop over list of `ScanPoint` × `repeats` |
| `teardown()` | End | Free tagger, disconnect stages |
| `with Acquisition(...) as acq:` | Preferred | `setup()` on enter, `teardown()` on exit |

## What gets saved?

```
base_dir/
  2026-06-19_14-30-00_raw/
    general_parameters.json    ← laser, channels, delays, ...
    2026-06-19_14-30-00_raw.log
    <stem>_chunk0.pkl          ← if using CorrelationRecorder
    data.ttbin                 ← if using RawTimeTagRecorder
    MERGED/
      <stem>_MERGED.pkl        ← if chunking enabled
```



---
# Part D — Minimal example (single short run)

This is the **smallest** working acquisition. Set `LIVE_HARDWARE = True` on the lab PC.


In [ ]:

from src.acquisition import Acquisition, AcquisitionConfig, ScanPoint, RawTimeTagRecorder
from src.hardware import RotationStageController

cfg = AcquisitionConfig(
    base_dir=ROOT / "data" / "tutorial_runs",
    material="TutorialSample",
    experiment_type="raw",
    chunking=ChunkingParams(enabled=False, chunk_minutes=0.5),
)

stages = None
if LIVE_HARDWARE:
    stages = RotationStageController([PRM1_SERIAL]).connect()

if LIVE_HARDWARE:
    with Acquisition(cfg, recorder=RawTimeTagRecorder(), stages=stages) as acq:
        point = ScanPoint(power_mw=10.0, angle_deg=50.0, stage_id=PRM1_SERIAL)
        acq.run_point(point, duration_s=5.0)   # 5 seconds
        print("Saved in:", acq.save_dir)
else:
    print("[offline] Would run 5 s raw acquisition into data/tutorial_runs/...")
    print("Code path: setup -> move stage -> record -> teardown")



---
# Part E — Chunked acquisition + early stop

When counts are low, you may need **hours** of data — but you don't know exactly how long.

**Chunking** = measure 2 min → save → measure 2 min → merge → check "enough yet?" → stop or continue.

The **stop condition** is a function `(merged_data) -> (should_stop, reason)`.

Built-in helper lives in `measurement.py`: `coincidence_threshold_stop(min_counts=100_000)`.

Enable chunking in config:
```python
ChunkingParams(enabled=True, chunk_minutes=2, max_chunks=100, stop_when_reached=True)
```


In [ ]:

from src.measurement import CorrelationRecorder, coincidence_threshold_stop

cfg_chunk = AcquisitionConfig(
    base_dir=ROOT / "data" / "tutorial_runs",
    material="CdTe110",
    experiment_type="g2_heralded_virtual",
    timetagging=TimeTaggerParams(coincidence_window_ps=25000),
    chunking=ChunkingParams(enabled=True, chunk_minutes=2, max_chunks=50),
)

stop_fn = coincidence_threshold_stop(min_counts=100_000, kind="physical")

if LIVE_HARDWARE:
    with Acquisition(
        cfg_chunk,
        recorder=CorrelationRecorder(),
        stop_condition=stop_fn,
    ) as acq:
        acq.run_chunked(ScanPoint(power_mw=42.0, label="P42"))
else:
    print("[offline] Chunk loop:")
    print("  for each chunk: record -> merge -> save MERGED.pkl -> check stop_fn")
    print("  stops when all physical coincidence pairs >= 100000 counts")



---
# Part F — Power / angle scan

`run_scan` loops:
```
for repeat in range(repeats):
    for point in points:
        move stage (if angle set)
        run_point OR run_chunked (depending on chunking.enabled)
```

Example: 3 powers at 3 angles = 9 acquisitions per repeat.


In [ ]:

powers = [10, 30, 50]
angles = [45, 55, 65]
points = [
    ScanPoint(power_mw=p, angle_deg=a, stage_id=PRM1_SERIAL)
    for p in powers for a in angles
]
print(f"Scan has {len(points)} points:", points[:2], "...")

if not LIVE_HARDWARE:
    print("[offline] acq.run_scan(points) executes all of them sequentially.")



---
# Summary

| You want… | Use… |
|-----------|------|
| Save every photon timestamp | `RawTimeTagRecorder()` |
| Save g² histograms live | `CorrelationRecorder()` (measurement tutorial) |
| One fixed-duration run | `run_point(point, duration_s=...)` |
| Run until enough coincidences | `chunking.enabled=True` + `stop_condition=` |
| Scan power & angle | `run_scan([ScanPoint(...), ...])` |

**Next:** `measurement_tutorial.ipynb` explains `CorrelationRecorder` and g² data in detail.
